# Cache BAG Building Data for Delft

This notebook fetches all building and address data from the BAG API for the entire city of Delft and saves it to pickle files for offline use.

**Output files:**
- `inputs/bag_cache/delft_all_buildings.pkl` - Raw building data (list of dicts)
- `inputs/bag_cache/delft_building_addresses.pkl` - Address data (dict)

These files can be used in `run_analysis.py` to avoid making BAG API calls.

In [1]:
import pandas as pd
import pickle
import os
import sys
from datetime import datetime
import geopandas as gpd
import folium
from shapely.geometry import Polygon

# Add parent directory to path to import project functions
sys.path.append('..')

from delft_calliope.functions.BAG_buildings_API import fetch_buildings_from_BAG
from delft_calliope.functions.BAG_addresses_API import enrich_buildings_with_addresses
from delft_calliope.functions.process_buildings import process_and_visualize_buildings

## Configuration

In [2]:
# BAG API Key
BAG_API_KEY = 'l7c0673beb4a3f46e8a0caa164dc7b8397'

# Bounding box for entire Delft (lon, lat in WGS84)
# Approximate bounds covering the whole city
DELFT_BBOX = [
    4.3388,   # min_lon (west)
    51.986,  # min_lat (south)
    4.37,   # max_lon (east)
    52.005   # max_lat (north)
]

# Output directory
OUTPUT_DIR = '../delft_calliope/inputs/bag_cache'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Delft bounding box: {DELFT_BBOX}")
print(f"Output directory: {OUTPUT_DIR}")

Delft bounding box: [4.3388, 51.986, 4.37, 52.005]
Output directory: ../delft_calliope/inputs/bag_cache


## Step 1: Fetch All Buildings in Delft

This may take several minutes depending on the number of buildings.

In [3]:
print(f"Starting building fetch at {datetime.now().strftime('%H:%M:%S')}")
print("This may take 5-15 minutes depending on the area size...\n")

all_buildings = fetch_buildings_from_BAG(DELFT_BBOX, BAG_API_KEY)

print(f"\nCompleted at {datetime.now().strftime('%H:%M:%S')}")
print(f"Total buildings fetched: {len(all_buildings)}")

Starting building fetch at 00:00:44
This may take 5-15 minutes depending on the area size...


Completed at 00:00:50
Total buildings fetched: 7719


## Step 2: Fetch Addresses for All Buildings

This will take longer - typically 10-30 minutes for a full city.

In [4]:
print(f"Starting address fetch at {datetime.now().strftime('%H:%M:%S')}")
print("This may take 10-30 minutes depending on the number of buildings...\n")

building_addresses = enrich_buildings_with_addresses(all_buildings, BAG_API_KEY)

print(f"\nCompleted at {datetime.now().strftime('%H:%M:%S')}")
print(f"Total buildings with addresses: {len(building_addresses)}")

Starting address fetch at 00:00:55
This may take 10-30 minutes depending on the number of buildings...


Completed at 00:04:49
Total buildings with addresses: 7490


## Step 3: Verify Data Structure

Check that the data has the expected structure.

In [5]:
# Check structure of all_buildings (list of dicts)
print("=" * 60)
print("Building data structure:")
print("=" * 60)
print(f"Type: {type(all_buildings)}")
print(f"Length: {len(all_buildings)}")
if len(all_buildings) > 0:
    print(f"\nFirst building keys: {list(all_buildings[0].keys())}")
    if 'pand' in all_buildings[0]:
        print(f"Pand keys: {list(all_buildings[0]['pand'].keys())}")

print("\n" + "=" * 60)
print("Address data structure:")
print("=" * 60)
print(f"Type: {type(building_addresses)}")
print(f"Length: {len(building_addresses)}")
if len(building_addresses) > 0:
    sample_id = list(building_addresses.keys())[0]
    print(f"\nSample entry:")
    print(f"  ID: {sample_id}")
    print(f"  Data: {building_addresses[sample_id]}")

Building data structure:
Type: <class 'list'>
Length: 7719

First building keys: ['pand', '_links']
Pand keys: ['identificatie', 'domein', 'geometrie', 'oorspronkelijkBouwjaar', 'status', 'geconstateerd', 'documentdatum', 'documentnummer', 'voorkomen']

Address data structure:
Type: <class 'dict'>
Length: 7490

Sample entry:
  ID: 0503100000000697
  Data: {'address': 'Wilhelminalaan 66', 'aantal_adressen': 1}


## Step 4: Save to Pickle Files

Save both datasets in their original format (not DataFrame) to preserve exact structure.

In [ ]:
# Save all_buildings as pickle (complex nested structure with geometry)
buildings_file = os.path.join(OUTPUT_DIR, 'delft_all_buildings.pkl')
with open(buildings_file, 'wb') as f:
    pickle.dump(all_buildings, f)
print(f"✓ Saved buildings to: {buildings_file}")
print(f"  File size: {os.path.getsize(buildings_file) / 1024 / 1024:.2f} MB")

# Also save addresses as pickle for exact compatibility
addresses_file = os.path.join(OUTPUT_DIR, 'delft_building_addresses.pkl')
with open(addresses_file, 'wb') as f:
    pickle.dump(building_addresses, f)
print(f"✓ Saved addresses to: {addresses_file} (pickle backup)")

✓ Saved buildings to: ../delft_calliope/inputs/bag_cache\delft_all_buildings.pkl
  File size: 5.12 MB

✓ Saved addresses to: ../delft_calliope/inputs/bag_cache\delft_building_addresses.csv
  File size: 0.23 MB
✓ Saved addresses to: ../delft_calliope/inputs/bag_cache\delft_building_addresses.pkl (pickle backup)


## Step 5: Test Loading from Cache

Verify the cached files can be loaded correctly.

In [7]:
# Load from cache
with open(buildings_file, 'rb') as f:
    loaded_buildings = pickle.load(f)

with open(addresses_file, 'rb') as f:
    loaded_addresses = pickle.load(f)

# Verify
print("Cache loading test:")
print(f" Loaded {len(loaded_buildings)} buildings")
print(f" Loaded {len(loaded_addresses)} building addresses")
print(f" Data types match: {type(loaded_buildings) == type(all_buildings)} and {type(loaded_addresses) == type(building_addresses)}")
print("\nCache is ready to use!")

Cache loading test:
 Loaded 7719 buildings
 Loaded 7490 building addresses
 Data types match: True and True

Cache is ready to use!


## Step 6: Create Summary Statistics

In [8]:
# Summary statistics
buildings_with_addresses = sum(1 for addr_info in building_addresses.values() if addr_info['aantal_adressen'] > 0)
buildings_without_addresses = len(building_addresses) - buildings_with_addresses
total_addresses = sum(addr_info['aantal_adressen'] for addr_info in building_addresses.values())

print("=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)
print(f"Total buildings fetched:        {len(all_buildings):,}")
print(f"Buildings with addresses:       {buildings_with_addresses:,}")
print(f"Buildings without addresses:    {buildings_without_addresses:,}")
print(f"Total addresses:                {total_addresses:,}")
print(f"Avg addresses per building:     {total_addresses / max(len(building_addresses), 1):.2f}")
print("=" * 60)

SUMMARY STATISTICS
Total buildings fetched:        7,719
Buildings with addresses:       4,388
Buildings without addresses:    3,102
Total addresses:                20,600
Avg addresses per building:     2.75


In [12]:
# Step 7: Test Cache with process_and_visualize_buildings

print("=" * 60)
print("TESTING CACHED DATA WITH EXISTING FUNCTION")
print("=" * 60)

# Load cached data
print("\n1. Loading cached pickle files...")
with open(buildings_file, 'rb') as f:
    cached_buildings = pickle.load(f)
    
with open(addresses_file, 'rb') as f:
    cached_addresses = pickle.load(f)

print(f"✓ Loaded {len(cached_buildings)} buildings")
print(f"✓ Loaded {len(cached_addresses)} building addresses")

# Create debug directory
print("\n2. Creating debug directory...")
debug_dir = '../delft_calliope/debug'
os.makedirs(debug_dir, exist_ok=True)
print(f"✓ Debug directory ready: {debug_dir}")

# Change to delft_calliope directory so relative paths work
print("\n3. Running process_and_visualize_buildings...")
original_dir = os.getcwd()
os.chdir('../delft_calliope')

try:
    buildings_df = process_and_visualize_buildings(
        cached_buildings, 
        cached_addresses, 
        mode='plot'
    )
    
    # Change back
    os.chdir(original_dir)
    
    print(f"✓ Successfully processed {len(buildings_df):,} buildings")
    print(f"✓ Map saved to: {os.path.abspath(os.path.join(debug_dir, 'buildings_addresses_map.html'))}")
    
    # Show sample of resulting dataframe
    print("\n4. Sample of buildings dataframe:")
    print(buildings_df.head())
    
    print("\n" + "=" * 60)
    print("✓ CACHE TEST SUCCESSFUL!")
    print("The cached files work perfectly with existing code!")
    print("=" * 60)
    
except Exception as e:
    os.chdir(original_dir)
    print(f"\n✗ Error: {e}")
    raise

TESTING CACHED DATA WITH EXISTING FUNCTION

1. Loading cached pickle files...
✓ Loaded 7719 buildings
✓ Loaded 7490 building addresses

2. Creating debug directory...
✓ Debug directory ready: ../delft_calliope/debug

3. Running process_and_visualize_buildings...
✓ Successfully processed 4,388 buildings
✓ Map saved to: c:\Users\alexn\Documents\GitHub\SC-G5\delft_calliope\debug\buildings_addresses_map.html

4. Sample of buildings dataframe:
                 id           addresses  nr_addresses year_construction  \
0  0503100000000697   Wilhelminalaan 66             1              1969   
1  0503100000000945       Gebbenlaan 46             1              1971   
2  0503100000002974  Van der Kamlaan 18             1              1969   
3  0503100000007170   Wilhelminalaan 98             1              1969   
4  0503100000007363       Gebbenlaan 50             1              1971   

                                            geometry           lon  \
0  POLYGON Z ((83031.945 444858.93 0

## Optional: Save Metadata

Save metadata about when this cache was created.

In [13]:
metadata = {
    'created_at': datetime.now().isoformat(),
    'bounding_box': DELFT_BBOX,
    'total_buildings': len(all_buildings),
    'buildings_with_addresses': buildings_with_addresses,
    'total_addresses': total_addresses
}

metadata_file = os.path.join(OUTPUT_DIR, 'cache_metadata.pkl')
with open(metadata_file, 'wb') as f:
    pickle.dump(metadata, f)
    
print(f" Saved metadata to: {metadata_file}")
print(f"\nAll files saved successfully!")

 Saved metadata to: ../delft_calliope/inputs/bag_cache\cache_metadata.pkl

All files saved successfully!
